# 06 - Synthetic data quality

Fidelity (KS, Wasserstein, correlation drift) and distance-based privacy proxies
(DCR) of every pool against the real training rows. CPU only. Run once over all
datasets. See `PIPELINE.md`, step 6.

In [ ]:
# Colab: upload df_train_*.csv, df_test_*.csv and the *_pool.csv files.
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    files.upload()

In [ ]:
import glob
import os
import warnings

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore", category=RuntimeWarning)
try:
    display
except NameError:
    display = print

## Config

In [ ]:
DATA_DIR = "."
DATASETS = ["diabetes", "pageblocks", "yeast", "ecoli"]   # those without files are skipped

SEED = 42
SAMPLE_PER_CLASS = 500    # rows per class for the marginal comparison
PRIVACY_SAMPLE   = 2000   # rows per side for DCR (identical rows use the full pool)
MAX_DISCRETE_LEVELS = 12

POOL_PATTERNS = {
    "cwgan":     "cwgan_{ds}_*_pool.csv",
    "ecwgan":    "ecwgan_E12345_{ds}_*_pool.csv",
    "ecwganVAE": "ecwganVAE_{ds}_*_pool.csv",
}

In [ ]:
def find_pools(ds):
    out = {}
    for label, pattern in POOL_PATTERNS.items():
        hits = sorted(glob.glob(os.path.join(DATA_DIR, pattern.format(ds=ds))))
        if len(hits) > 1:
            raise ValueError(f"{ds}/{label}: more than one pool found, keep one: {hits}")
        if hits:
            out[label] = hits[0]
    return out


DATASETS = [ds for ds in DATASETS
            if os.path.isfile(os.path.join(DATA_DIR, f"df_train_{ds}.csv"))
            and os.path.isfile(os.path.join(DATA_DIR, f"df_test_{ds}.csv"))]
POOLS = {ds: find_pools(ds) for ds in DATASETS}
for ds in DATASETS:
    print(ds, "->", {k: os.path.basename(v) for k, v in POOLS[ds].items()})

## Measures

In [ ]:
def prepare(ds):
    df_train = pd.read_csv(os.path.join(DATA_DIR, f"df_train_{ds}.csv"))
    df_test = pd.read_csv(os.path.join(DATA_DIR, f"df_test_{ds}.csv"))
    target = df_train.columns[-1]
    df_test = df_test[df_train.columns]
    features = [c for c in df_train.columns if c != target]
    discrete = [c for c in features if df_train[c].nunique() <= MAX_DISCRETE_LEVELS]
    continuous = [c for c in features if c not in discrete]
    return dict(target=target, features=features, continuous=continuous,
                counts=df_train[target].value_counts().sort_index(),
                df_train=df_train, df_test=df_test)


def stratified(df, n, ctx):
    return pd.concat([g.sample(min(n, len(g)), random_state=SEED)
                      for _, g in df.groupby(ctx["target"])]).reset_index(drop=True)


def matched(pool, ctx):
    """The pool cut to the real class counts, so marginals share one class mix."""
    parts = []
    for cls, n in ctx["counts"].items():
        p = pool[pool[ctx["target"]] == cls]
        parts.append(p.iloc[:min(int(n), len(p))])
    return pd.concat(parts, ignore_index=True)


def marginals(real, synth, ctx):
    """Mean KS statistic and Wasserstein distance over the continuous columns."""
    ks, wd = [], []
    for c in ctx["continuous"]:
        a, b = real[c].dropna(), synth[c].dropna()
        if len(a) > 1 and len(b) > 1:
            ks.append(ks_2samp(a, b).statistic)
            wd.append(wasserstein_distance(a, b))
    return (float(np.mean(ks)) if ks else np.nan, float(np.mean(wd)) if wd else np.nan)


def corr_drift(real, synth, ctx):
    """Mean absolute difference of the correlation matrices; constant columns are skipped."""
    d = np.abs(real[ctx["features"]].corr().to_numpy() - synth[ctx["features"]].corr().to_numpy())
    vals = d[np.triu_indices_from(d, k=1)]
    vals = vals[~np.isnan(vals)]
    return float(vals.mean()) if len(vals) else np.nan


def dcr(real, synth, ctx):
    """5th percentile of the distance to the closest real record (2,000-row samples)."""
    rng = np.random.RandomState(SEED)
    n = PRIVACY_SAMPLE
    r = real if len(real) <= n else real.iloc[rng.choice(len(real), n, replace=False)]
    s = synth if len(synth) <= n else synth.iloc[rng.choice(len(synth), n, replace=False)]
    nn_ = NearestNeighbors(n_neighbors=2).fit(r[ctx["features"]].to_numpy())
    d1 = nn_.kneighbors(s[ctx["features"]].to_numpy())[0][:, 0]
    return float(np.percentile(d1, 5))


def identical_rows(real, synth, ctx):
    """Generated rows whose features exactly equal a training row, over the full pool."""
    seen = set(pd.util.hash_pandas_object(real[ctx["features"]], index=False))
    return int(pd.util.hash_pandas_object(synth[ctx["features"]], index=False).isin(seen).sum())

## Run

In [ ]:
rows = []
for ds in DATASETS:
    ctx = prepare(ds)
    real_s = stratified(ctx["df_train"], SAMPLE_PER_CLASS, ctx)
    ref_dcr = dcr(ctx["df_train"], ctx["df_test"], ctx)        # real-vs-real reference
    for label, path in POOLS[ds].items():
        pool = pd.read_csv(path)[ctx["df_train"].columns]
        synth = matched(pool, ctx)                             # fidelity on the matched subset
        ks, wd = marginals(real_s, stratified(synth, SAMPLE_PER_CLASS, ctx), ctx)
        p5 = dcr(ctx["df_train"], pool, ctx)
        identical = identical_rows(ctx["df_train"], pool, ctx)   # checked on the full pool
        rows.append({"dataset": ds, "pool": label, "pool rows": len(pool),
                     "KS": ks, "Wasserstein": wd,
                     "corr drift": corr_drift(ctx["df_train"], synth, ctx),
                     "DCR p5": p5, "DCR p5 real-vs-real": ref_dcr,
                     "identical rows": identical})
        del pool, synth

quality = pd.DataFrame(rows)
display(quality.round(4))

In [ ]:
OUT_FILE = os.path.join(DATA_DIR, "quality_all.csv")
quality.to_csv(OUT_FILE, index=False)
print("written:", OUT_FILE)
if IN_COLAB:
    files.download(OUT_FILE)